# Quarta prova intermedia · Spark Structured Streaming

1. Generare corse con la sorgente `rate` e archiviarle in Parquet su HDFS.
2. Calcolare conteggi e quote di mercato per categoria in finestre tumbling da un minuto.
3. Calcolare ricavo medio per corsa e per miglio in finestre sliding da due minuti, con passo di 30 secondi.
4. Fermare la generazione, drenare le query e confrontare gli indicatori con Spark SQL batch sugli stessi Parquet.

## Configurazione

Imposta `RUN_MODE = "run"` per avviare una simulazione oppure `RUN_MODE = "demo"` per riaprire risultati già salvati. La modalità demo con `RUN_ID = None` seleziona l'ultima run disponibile. Gli orari sono in UTC.

In [ ]:
from src.config import LabConfig, RunPaths, list_runs, spark_session

RUN_MODE = "demo"
RUN_ID = None
INSPECT_HDFS = False
CONFIG = LabConfig(
    duration_seconds=600,
    rows_per_second=10,
    trigger_seconds=10,
    watermark_seconds=120,
    live_seconds=25,
)

if RUN_MODE not in {"run", "demo"}:
    raise ValueError('RUN_MODE deve essere "run" o "demo".')

if RUN_MODE == "demo":
    available_runs = list_runs()
    selected_run = RUN_ID or next(iter(available_runs), None)
    print("Run disponibili:", ", ".join(available_runs) or "nessuna")
else:
    print(CONFIG)

## 1 · Dalla sorgente `rate` a HDFS

`value` fornisce l'identificativo e `timestamp` l'istante di inizio della corsa. Due hash Spark del gruppo di dieci ID determinano una permutazione pseudo-casuale delle posizioni del gruppo. Le posizioni 0–5 assegnano HVFHV, 6–8 Yellow e 9 Green. Ogni gruppo completo contiene 6, 3 e 1 corse rispettivamente, con un ordine riproducibile che varia tra i gruppi.

Due hash Spark deterministici producono la distanza (0,500–25,000 miglia) e la variazione della durata (−90…330 secondi). La durata parte da 180 secondi più circa 165 secondi per miglio, con la variazione applicata e il totale limitato a 180–5.400 secondi.

| Tipo | Quota fissa (€) | €/miglio | €/minuto |
|---|---:|---:|---:|
| HVFHV | 3,50 | 2,25 | 0,45 |
| Yellow | 3,00 | 2,50 | 0,50 |
| Green | 2,75 | 2,35 | 0,48 |

Le tariffe sono simulate. Si sommano la quota fissa e le due componenti variabili, arrotondate ciascuna al centesimo. L'archivio conserva distanza, durata e ricavo come interi.

Un solo writer in modalità `append` archivia le corse in `trips/`. Entrambe le query leggono i Parquet da quella directory con lo schema canonico. Ogni query ha un proprio checkpoint.

## 2 · Tumbling · corse e quota di mercato

Ogni corsa appartiene a una finestra da un minuto `[inizio, fine)`. I conteggi condizionali per HVFHV, Yellow e Green e il totale vengono calcolati nella stessa aggregazione. La quota è `100 × corse categoria / corse totali` nella finestra. Le categorie assenti hanno conteggio e quota pari a zero.

La modalità `update` pubblica le revisioni delle finestre. Gli snapshot conservano i bordi della finestra, il numero di micro-batch e il momento di elaborazione in UTC.

## 3 · Sliding · ricavi per corsa e per miglio

Una corsa appartiene a quattro finestre da due minuti, con inizio ogni 30 secondi. Si sommano i centesimi e i millesimi di miglio per categoria. Il ricavo medio per corsa è `centesimi / (100 × corse)`. Quello per miglio è `10 × centesimi / millesimi di miglio`, il rapporto tra i due totali.

L’*event time* è l’inizio della corsa. Il *processing time* indica quando Spark elabora il micro-batch. Il watermark della query sliding avanza con l’event time osservato, con un ritardo configurabile di 120 secondi. Questo consente a Spark di rilasciare lo stato delle finestre vecchie. Dati troppo tardivi possono essere scartati. Il log espone il watermark e gli scarti riportati da Spark. Le finestre parziali ai bordi della run restano nel confronto.

## Aggiornamenti durante la simulazione

La vista si aggiorna ogni 25 secondi circa. Mostra il numero di corse pubblicate su HDFS, gli ultimi micro-batch confermati e alcune revisioni delle finestre. L’ora di trigger, l’ora della rilevazione e l’inizio della corsa descrivono momenti diversi.

In [ ]:
from IPython.display import HTML, clear_output, display
from matplotlib import pyplot as plt
from src.live import table_html, trend_figure


def render_live(view):
    clear_output(wait=True)
    print(f"Run {view['run_id']} · corse archiviate: {view['archived_rows']}")
    print(f"Watermark UTC: {view['watermark_utc'] or 'in attesa'} · "
          f"righe scartate da Spark: {view['rows_dropped_by_watermark']}")
    display(HTML(table_html(view["progress"])))
    print("Ultime finestre tumbling")
    display(HTML(table_html(view["tumbling"][-3:])))
    print("Ultime finestre sliding")
    display(HTML(table_html(view["sliding"][-6:])))
    fig = trend_figure(view["tumbling"], view["sliding"])
    display(fig)
    plt.close(fig)

## 4 · Arresto, drenaggio e confronto

Il target è la durata nominale moltiplicata per la frequenza. Con i valori predefiniti si archiviano gli ID da 0 a 5.999. Dopo la conferma dei micro-batch, Spark rilegge il Parquet su HDFS e verifica conteggio, unicità ed estremi degli ID. Solo allora ferma il generatore e drena i due lettori. Il manifest registra il target, il conteggio effettivo e il tempo misurato dall’avvio del generatore alla verifica dell’archivio, prima del drenaggio. Un’interruzione o un timeout lascia la run incompleta.

### Ricalcolo Spark SQL

Le query batch leggono la directory Parquet della run dopo il drenaggio. Usano gli stessi confini temporali e tengono tutte le finestre, incluse quelle parziali. Gli indicatori batch vengono scritti su HDFS e in CSV.

In [ ]:
from IPython.display import Markdown
from src.batch import SLIDING_SQL, TUMBLING_SQL

display(Markdown("#### Tumbling\n```sql\n" + TUMBLING_SQL + "\n```"))
display(Markdown("#### Sliding\n```sql\n" + SLIDING_SQL + "\n```"))

### Confronto per finestra e categoria

Si conserva l’ultima revisione di ogni chiave in base al `batch_id`. Il full outer join evidenzia chiavi mancanti su entrambi i lati e attribuisce `value_mismatch` ai valori discordanti. I conteggi e le somme intere devono essere identici. Per quote e medie degli importi vale una tolleranza assoluta di `1e-6`. I controlli globali comprendono conteggi, quote, unicità degli ID e scarti da watermark.

In [ ]:
from src.comparison import run_lab

if RUN_MODE == "run":
    spark = spark_session()
    paths, manifest, archived_trips = run_lab(spark, CONFIG, on_update=render_live)

## Esito e modalità demo

La demo carica manifest, tabelle, tendenze e log dalla directory locale della run. Imposta `INSPECT_HDFS = True` per consultare anche le corse Parquet e il confronto su HDFS. I CSV conservano tutti i valori e le chiavi del confronto.

In [ ]:
from IPython.display import Image
from pyspark.sql import functions as F
from src.demo import read_csv, read_manifest, read_progress


def preview(rows, columns, limit=9):
    return [
        {
            key: float(row[key]) if key.startswith(("share_", "avg_")) and row.get(key) else row.get(key)
            for key in columns
        }
        for row in rows[:limit]
    ]


if RUN_MODE == "demo":
    paths = RunPaths(selected_run) if selected_run else None
    if paths:
        manifest = read_manifest(paths)
        if INSPECT_HDFS:
            spark = spark_session()
            archived_trips = spark.read.parquet(paths.trips)

if paths is None:
    print('Per generare dati, imposta RUN_MODE = "run" nella configurazione ed esegui tutte le celle.')
else:
    print(f"Run {paths.run_id} · {manifest['status']} · confronto {manifest.get('comparison') or 'non concluso'}")
    print(f"Avvio UTC {manifest['started_at_utc']} · fine UTC {manifest.get('finished_at_utc') or '—'}")
    if manifest.get("error"):
        print(manifest["error"])

    total = manifest.get("trip_count") or 0
    target = manifest.get("target_trip_count")
    if target is not None:
        elapsed = manifest.get("generation_duration_seconds")
        duration_text = f"{elapsed:.1f} s" if elapsed is not None else "—"
        print(f"Corse attese: {target} · durata reale della generazione: {duration_text}")
    print(f"Corse archiviate: {total} · ID duplicati: {manifest.get('duplicate_ids', '—')} · "
          f"scarti da watermark: {manifest.get('rows_dropped_by_watermark', '—')}")
    print("Finestre parziali:", manifest.get("partial_windows", {}))
    if total and manifest.get("category_counts"):
        display(HTML(table_html([
            {"Categoria": name, "Corse": count, "Quota (%)": 100 * count / total}
            for name, count in manifest["category_counts"].items()
        ])))

    if manifest.get("checks"):
        display(HTML(table_html([
            {"Controllo": name, "Esito": "PASS" if ok else "FAIL"}
            for name, ok in manifest["checks"].items()
        ])))
        print("Chiavi per esito:", manifest["comparison_rows"], "· differenze:", manifest["differences"])

    tumbling_rows = read_csv(paths, "streaming_tumbling")
    sliding_rows = read_csv(paths, "streaming_sliding")
    if tumbling_rows:
        print("Quote per categoria · ultime finestre tumbling")
        display(HTML(table_html(preview(tumbling_rows[-5:], (
            "window_start", "n_total", "n_hvfhv", "n_yellow", "n_green",
            "share_hvfhv", "share_yellow", "share_green",
        )))))
    if sliding_rows:
        print("Ricavi per categoria · ultime finestre sliding")
        display(HTML(table_html(preview(sliding_rows[-9:], (
            "window_start", "vehicle_type", "n", "avg_revenue_per_trip_eur", "avg_revenue_per_mile_eur",
        )))))

    chart = paths.local / "trend.png"
    if chart.is_file():
        display(Image(filename=str(chart)))
    history = read_progress(paths)
    if history:
        print("Ultimi micro-batch. Il registro completo è in progress.jsonl.")
        display(HTML(table_html(preview(history[-8:], (
            "query", "batch_id", "trigger_at_utc", "observed_at_utc",
            "input_rows", "event_max_utc", "watermark_utc", "rows_dropped_by_watermark",
        )))))

    diagnostics = read_csv(paths, "comparison")
    if diagnostics:
        problems = [row for row in diagnostics if row["result"] != "PASS"]
        print("Diagnostica ·", "prime difformità" if problems else "prime chiavi")
        display(HTML(table_html(preview(problems or diagnostics, (
            "kind", "window_start", "window_end", "vehicle_type", "result",
            "partial_window", "metrics_mismatched",
        )))))
        print(f"Confronto completo: results/{paths.run_id}/comparison.csv")

    if RUN_MODE == "run" or INSPECT_HDFS:
        print("Campione di corse dal Parquet")
        archived_trips.orderBy("trip_id").select(
            "trip_id", "start_time", "vehicle_type",
            (F.col("distance_milli_miles") / 1000).alias("distance_miles"),
            (F.col("duration_seconds") / 60).alias("duration_minutes"),
            (F.col("revenue_cents") / 100).alias("revenue_eur"),
        ).show(6, truncate=False)
        if RUN_MODE == "demo" and manifest.get("comparison"):
            spark.read.parquet(paths.final("comparison")).groupBy("kind", "result").count().show()

### Conclusioni

Il confronto verifica gli indicatori prodotti dallo streaming sulle stesse corse archiviate e rilette da Spark SQL. L’esito riguarda tutte le chiavi dell’esecuzione, comprese le finestre parziali. In caso di `FAIL`, il CSV diagnostico riporta i valori dei due lati e le metriche discordanti.